# v61 Shadow Residual Primitive Modular Derivation

No file IO. Each code cell checks one closure claim and prints PASS/FAIL with numeric data.

In [ ]:
import math, cmath
import numpy as np
N=12
support={1:1,5:-1,7:-1,11:1}
def chi12(r): return support.get(r%12,0)
def S():
    return np.array([[cmath.exp(-math.pi*1j*r*s/6)/math.sqrt(12) for s in range(N)] for r in range(N)], dtype=complex)
print('setup: PASS')

## Claim 1: Q square seating gives the T phase on the Shadow support.

In [ ]:
rows=[(r,(r*r)%24,chi12(r)) for r in range(N)]
passed=all((r*r)%24==1 for r in support)
print('T square seating:', 'PASS' if passed else 'FAIL')
print([row for row in rows if row[2]!=0])

## Claim 2: B/L finite dual seating gives a unitary S kernel.

In [ ]:
F=S()
err=float(np.max(np.abs(F.conjugate().T@F-np.eye(N))))
print('S unitarity:', 'PASS' if err<1e-12 else 'FAIL', 'max_err=', err)

## Claim 3: S squared is orientation reversal.

In [ ]:
F=S(); C=np.zeros((N,N), dtype=complex)
for r in range(N): C[r,(-r)%N]=1
err=float(np.max(np.abs(F@F-C)))
print('S^2 reversal:', 'PASS' if err<1e-12 else 'FAIL', 'max_err=', err)

## Claim 4: chi12 is preserved by the finite S action.

In [ ]:
F=S(); chi=np.array([chi12(r) for r in range(N)], dtype=complex)
err=float(np.max(np.abs(F@chi-chi)))
print('chi S eigen:', 'PASS' if err<1e-12 else 'FAIL', 'max_err=', err)
print(np.round(F@chi, 12))

## Claim 5: coefficient stripping gives eta through the tested window.

In [ ]:
max_power=300
eta=[0]*(max_power+1)
for n in range(1, int(math.sqrt(24*max_power+1))+2):
    c=chi12(n)
    if c:
        m=(n*n-1)//24
        if n*n==24*m+1 and 0<=m<=max_power: eta[m]+=c
pent=[0]*(max_power+1)
for k in range(-1000,1001):
    m=k*(3*k-1)//2
    if 0<=m<=max_power: pent[m]+=(-1)**k
mism=[(i,eta[i],pent[i]) for i in range(max_power+1) if eta[i]!=pent[i]]
print('eta coefficient collapse:', 'PASS' if not mism else 'FAIL', 'mismatch_count=', len(mism))
print('first_nonzero=', [(i,c) for i,c in enumerate(eta) if c][:12])

## Claim 6: scalar projection cancels the retained n-weighted orientation channel.

In [ ]:
rows=[]
for n in range(1,80):
    c=chi12(n)
    if c: rows.append((n,c,c*n,-c*n,c*n-c*n))
passed=all(row[-1]==0 for row in rows)
print('orientation scalar projection:', 'PASS' if passed else 'FAIL')
print(rows[:12])

## Claim 7: derivative channel is the weight-3/2 retained channel.

In [ ]:
# Finite generator check: the derivative channel has the same finite S kernel with the additional orientation factor i.
F=S(); S_three=1j*F
err=float(np.max(np.abs(S_three/1j - F)))
print('derivative finite kernel lift:', 'PASS' if err<1e-15 else 'FAIL', 'max_err=', err)
print('S_three[1,5]=', S_three[1,5])